In [19]:
from ikpy.chain import Chain
from scipy.spatial.transform import Rotation as R
import numpy as np
import os

# ✅ Jupyter에서는 __file__ 대신 os.getcwd() 사용
urdf_path = os.path.join(os.getcwd(), '../config', 'gazebo_jetcobot.urdf')

# 체인 로딩
jetcobot_chain = Chain.from_urdf_file(urdf_path)

# 목표 pose 정의 (위치 + RPY → 4x4 행렬)
pos = [-155.4, -62.3, 350.6]
rpy = [1.85, -0.67, -91.6]
r_rpy = np.radians(rpy)
rot = R.from_euler('xyz', r_rpy).as_matrix()

target_pose_matrix = np.eye(4)
target_pose_matrix[:3, :3] = rot
target_pose_matrix[:3, 3] = pos

# ✅ IK 계산 (주의: frame version 사용!)
joint_angles = jetcobot_chain.inverse_kinematics_frame(target_pose_matrix)
deg_jnt_angs = np.degrees(joint_angles)
print("Joint Angles:", deg_jnt_angs)


Joint Angles: [  0.         -20.15027407  19.39306202   0.36023502  52.81070304
 -20.13664664  -4.38870781   0.        ]


/home/addinedu/venv/pymycobot/lib/python3.12/site-packages/ikpy/urdf/URDF.py:261: UserWarning: Joint jiazhua_Joint is of type: fixed, but has an 'axis' attribute defined. This is not in the URDF spec and thus this axis is ignored
  warnings.warn("Joint {} is of type: fixed, but has an 'axis' attribute defined. This is not in the URDF spec and thus this axis is ignored".format(joint.attrib["name"]))
/home/addinedu/venv/pymycobot/lib/python3.12/site-packages/ikpy/chain.py:60: UserWarning: Link Base link (index: 0) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive
  warnings.warn("Link {} (index: {}) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive".format(link.name, link_index))
/home/addinedu/venv/pymycobot/lib/python3.12/site-packages/ikpy/chain.py:60: UserWarning: Link jiazhua_Join

In [34]:
from ikpy.chain import Chain
import numpy as np


def get_pose_of_link(chain, joint_angles, target_link_index):
    # 모든 링크의 pose를 순서대로 반환
    transforms = chain.forward_kinematics(joint_angles, full_kinematics=True)
    return transforms[target_link_index]



# 체인 로딩
urdf_path = os.path.join(os.getcwd(), '../config', 'gazebo_jetcobot.urdf')
jetcobot_chain = Chain.from_urdf_file(urdf_path)

# 각도 예시 (라디안)
# joint_angles = [0.0] * len(jetcobot_chain.active_links_mask)
joint_angles = [0.015, 0.006, -0.011, -0.015, -0.014, -0.77, 0.0, 0.0]

# 링크 리스트 확인
for idx, link in enumerate(jetcobot_chain.links):
    print(f"[{idx}] {link.name}")

index_6_link = next(i for i, l in enumerate(jetcobot_chain.links) if l.name == '6_Joint')
pose_6_link = get_pose_of_link(jetcobot_chain, joint_angles, index_6_link)
print("Pose of 6_Joint:\n", pose_6_link)

[0] Base link
[1] 1_Joint
[2] 2_Joint
[3] 3_Joint
[4] 4_Joint
[5] 5_Joint
[6] 6_Joint
[7] jiazhua_Joint
Pose of 6_Joint:
 [[ 7.21500455e-01  6.91258544e-01  3.99839934e-02  3.00657438e-02]
 [-6.91818736e-01  7.22071172e-01  2.41732732e-04 -8.51890539e-02]
 [-2.87041892e-02 -2.78360860e-02  9.99200291e-01  4.14255566e-01]
 [ 0.00000000e+00  0.00000000e+00  0.00000000e+00  1.00000000e+00]]


/home/addinedu/venv/pymycobot/lib/python3.12/site-packages/ikpy/urdf/URDF.py:261: UserWarning: Joint jiazhua_Joint is of type: fixed, but has an 'axis' attribute defined. This is not in the URDF spec and thus this axis is ignored
  warnings.warn("Joint {} is of type: fixed, but has an 'axis' attribute defined. This is not in the URDF spec and thus this axis is ignored".format(joint.attrib["name"]))
/home/addinedu/venv/pymycobot/lib/python3.12/site-packages/ikpy/chain.py:60: UserWarning: Link Base link (index: 0) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive
  warnings.warn("Link {} (index: {}) is of type 'fixed' but set as active in the active_links_mask. In practice, this fixed link doesn't provide any transformation so is as it were inactive".format(link.name, link_index))
/home/addinedu/venv/pymycobot/lib/python3.12/site-packages/ikpy/chain.py:60: UserWarning: Link jiazhua_Join

In [16]:
# 기본자세 joint angles (모두 0)
zero_angles = [0] * len(jetcobot_chain.active_links_mask)

# FK로 End-effector의 pose 구함
ee_pose = jetcobot_chain.forward_kinematics(zero_angles)

# 다시 이 pose를 target으로 넣어보면?
ik_result = jetcobot_chain.inverse_kinematics_frame(ee_pose)

print("IK Result (from FK identity):", ik_result)


IK Result (from FK identity): [0. 0. 0. 0. 0. 0. 0. 0.]


In [17]:
zero_angles = [0.0] * len(jetcobot_chain.active_links_mask)
pose = jetcobot_chain.forward_kinematics(zero_angles)
print("End-Effector Pose at Zero Angles:\n", pose)

End-Effector Pose at Zero Angles:
 [[ 1.          0.          0.          0.09581993]
 [ 0.          1.          0.         -0.06318034]
 [ 0.          0.          1.          0.40753739]
 [ 0.          0.          0.          1.        ]]
